In [1]:
import importlib
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import nltk
from nltk.stem import WordNetLemmatizer

import NLPPrepocessing
importlib.reload(NLPPrepocessing)
from NLPPrepocessing import NLPPreprocessing

## Laden der Daten über das Skript + Lemmatization

In [2]:
dataFrame = pd.read_csv("resume_data.csv") 

preProcesser = NLPPreprocessing(dataFrame, dataFrame.columns, 4000)
X_train, X_test, y_train, y_test = preProcesser.preprocess_and_split(0.2)

# Einmaliger Download der Wörterbücher (läuft beim zweiten Mal in Millisekunden durch)
nltk.download('wordnet')
nltk.download('omw-1.4') 

# Lemmatization (Wörter auf Grundform zurückführen)
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    # Nimmt jedes Wort, sucht es im Wörterbuch und gibt die Grundform zurück
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

# Wende die Lemmatization auf die Trainings- und Testdaten an
X_train['candidate_consolidated'] = X_train['candidate_consolidated'].apply(lemmatize_text)
X_train['job_consolidated'] = X_train['job_consolidated'].apply(lemmatize_text)
X_test['candidate_consolidated'] = X_test['candidate_consolidated'].apply(lemmatize_text)
X_test['job_consolidated'] = X_test['job_consolidated'].apply(lemmatize_text)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/stefanhromada/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/stefanhromada/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## TfidfVectorizer

In [3]:
# Um einen gemeinsamen Vektorraum zu haben, trainieren wir den Vectorizer 
# auf dem Vokabular sowohl der Kandidaten als auch der Jobs aus den Trainingsdaten.
corpus_train = X_train['candidate_consolidated'].tolist() + X_train['job_consolidated'].tolist()

# TfidfVectorizer initialisieren (Stoppwörter entfernen hilft bei reinem Keyword-Matching)
tfidf = TfidfVectorizer(stop_words='english', lowercase=True)
tfidf.fit(corpus_train)

# Transformieren der Testdaten in Sparse-Vektoren
candidate_tfidf_test = tfidf.transform(X_test['candidate_consolidated'])
job_tfidf_test = tfidf.transform(X_test['job_consolidated'])

## Cosine Distances

In [4]:
# Wir nutzen paired_cosine_distances für effizienten paarweisen Vergleich 
y_pred_baseline = 1 - paired_cosine_distances(candidate_tfidf_test, job_tfidf_test)

## Baseline Performance Report

In [5]:
# TEIL 4: Deliverable: Baseline Performance Report
mse = mean_squared_error(y_test, y_pred_baseline)
pearson_corr, p_value = pearsonr(y_test, y_pred_baseline)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Pearson Correlation:      {pearson_corr:.4f} (p-value: {p_value:.4e})")

 BASELINE PERFORMANCE REPORT (STEMMED)
Mean Squared Error (MSE): 0.1363
Pearson Correlation:      0.1459 (p-value: 1.4943e-10)


In [6]:
analysis_df = X_test.copy()
analysis_df['actual_score'] = y_test.values
analysis_df['predicted_tfidf_score'] = y_pred_baseline
analysis_df['error'] = abs(analysis_df['actual_score'] - analysis_df['predicted_tfidf_score'])

# Wir holen uns alle Wörter, die der TF-IDF Vectorizer gelernt hat
feature_names = tfidf.get_feature_names_out()

In [7]:
def analyze_and_print_overlap(df_subset, title, num_results=5):
    print(f"{title}: {len(df_subset)}")
    
    for index, row in df_subset.head(num_results).iterrows():
        print(f"KANDIDATENTEXT (Lemmatized):\n{row['candidate_consolidated']}\n")
        print(f"JOBTEXT (Lemmatized):\n{row['job_consolidated']}\n")
        print(f"▶ Tatsächlicher Score: {row['actual_score']:.2f}  |  Vorhersage (TF-IDF): {row['predicted_tfidf_score']:.2f}\n")
        
        cand_vec = tfidf.transform([row['candidate_consolidated']])
        job_vec = tfidf.transform([row['job_consolidated']])
        
        cand_indices = cand_vec.nonzero()[1]
        job_indices = job_vec.nonzero()[1]
        
        overlap_indices = set(cand_indices).intersection(set(job_indices))
        overlapping_words = [feature_names[i] for i in overlap_indices]
        
        if overlapping_words:
            print(f"▶ Überlappende TF-IDF Wörter:\n{', '.join(overlapping_words)}")
        else:
            print("▶ Überlappende TF-IDF Wörter:\nKEINE (0 Überschneidungen)")

In [8]:
# 1. Mismatches filtern
mismatches = analysis_df[
    (analysis_df['actual_score'] > 0.7) &  
    (analysis_df['predicted_tfidf_score'] < 0.3)
].sort_values(by='error', ascending=False)

# 2. Matches filtern
matches = analysis_df[
    (analysis_df['actual_score'] > 0.7) &  
    (analysis_df['predicted_tfidf_score'] > 0.7)
].sort_values(by='error', ascending=False)

# 3. Funktion für beide Datensätze aufrufen
analyze_and_print_overlap(mismatches, title="Anzahl gefundener extremer Mismatches")
analyze_and_print_overlap(matches, title="Anzahl gefundener Matches")



######################################################################
Anzahl gefundener extremer Mismatches: 309
######################################################################

KANDIDATENTEXT (Lemmatized):
objective: financial and accounting professional with expertise in financial analysis, audit, compliance, financial accounting, forecasting, budgeting, and procurement in the healthcare industry. strong working knowledge of gaap, fasb, sox, and asc 606 procedures. exceptional analytical ability and problem-solving skills, analysis and solution of complex problem in conjunction with managing financial outputs, managing team to achieve defined outcome with over 15 years' experience in a variety of organizational roles. advanced knowledge of microsoft toolsets. result oriented with proven track record of quick learning ability, increased responsibility and rapid advancement. | skills: power user of microsoft excel epicor netsuite quickbooks hyperion great plain sage intacct, 